In [ ]:
# 加载根目录
import os

# 直接指定项目根目录的绝对路径
project_root = "d:\\project\\storySummary"
os.chdir(project_root)
print(f"工作目录: {os.getcwd()}")

# 加载配置
from src.config import config

In [ ]:
# 节点生成测试
import os
# 直接指定项目根目录的绝对路径
project_root = "d:\\project\\storySummary"
os.chdir(project_root)
print(f"工作目录: {os.getcwd()}")

# 加载配置
from src.config import config


import pathlib
from src.services.book_service import BookService
from src.storage.database import Database
from src.pipeline import NovelToPodcastPipeline

# 确保使用正确的路径分隔符
book_path = pathlib.Path("samples") / "夏日沉沦纳粹高徒.txt"

# 检查文件是否存在
print(f"文件是否存在: {book_path.exists()}")
print(f"文件路径: {book_path.absolute()}")

# 1. 使用 BookService 处理文件
book_service = BookService()
result = await book_service.process_file(str(book_path), user_id="test_user03",max_chunk_chars=10000)
book_id = result["book_id"]
title = result["title"]
story_chunks = result["story_chunks"]
print(f"文件处理完成: book_id={book_id}, title={title}")

# 2. 使用 NovelToPodcastPipeline 生成节点
pipeline = NovelToPodcastPipeline(
    db_path=config.DATABASE_PATH, 
    vector_store_path=config.VECTOR_DB_DIR, 
    api_key=config.DEEPSEEK_API_KEY
)
process_result = await pipeline.process(story_chunks, title, book_id)
print(f"节点生成完成: {len(process_result['nodes'])} 个节点")

In [ ]:
# 加载根目录
import os

# 直接指定项目根目录的绝对路径
project_root = "d:\\project\\storySummary"
os.chdir(project_root)

from src.storage.manuscript_repository import manuscript_repository
from src.storage.book_repository import book_repository
from src.generation.pipeline import ManuscriptPipeline
BOOK_ID = "030981a1-bd74-4cd2-97b4-545959eed228"

style = manuscript_repository.load_style_profile(book_id=BOOK_ID)
outline = manuscript_repository.load_outline(book_id=BOOK_ID)
chunks = book_repository.load_chunks(book_id=BOOK_ID)

reference_file = os.path.join("samples", "product","《神的九十亿个名字》.txt")
with open(reference_file, 'r', encoding='utf-8') as f:
    reference_text = f.read()

def progress_callback(number,progress):
    print(f"当前进度: {progress}")
pipeline = ManuscriptPipeline(progress_callback=progress_callback,reference_script=reference_text)
res = await pipeline.run(book_id=BOOK_ID)
print("pipeline complete")